In [1]:
# TEST DI SANITÀ: Verifica Data Leakage nella Normalizzazione

import numpy as np
from sklearn.model_selection import train_test_split
from data_handler.data_loader import normalize, load_cup, load_monk

print("="*70)
print("TEST 1: Verifica Data Leakage con Dataset CUP")
print("="*70)

# Carica dataset (usa il tuo path)
X, y = load_cup('data/CUP/ML-CUP25-TR.csv', training=True)

# Statistiche PRIMA della normalizzazione
print(f"\n📊 DATASET ORIGINALE (prima di split e normalizzazione)")
print(f"   X shape: {X.shape}")
print(f"   X mean (feature 0): {X[:, 0].mean():.6f}")
print(f"   X std (feature 0):  {X[:, 0].std():.6f}")
print(f"   X min/max: [{X.min():.3f}, {X.max():.3f}]")

# Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)

print(f"\n📦 DOPO SPLIT (ancora non normalizzato)")
print(f"   X_train shape: {X_train.shape}")
print(f"   X_val shape:   {X_val.shape}")
print(f"   X_train mean (feature 0): {X_train[:, 0].mean():.6f}")
print(f"   X_val mean (feature 0):   {X_val[:, 0].mean():.6f}")

# Normalizzazione CORRETTA (fit su train, transform su val)
X_train_norm, mean_x, std_x = normalize(X_train)
X_val_norm, _, _ = normalize(X_val, mean=mean_x, std=std_x)

print(f"\n✅ DOPO NORMALIZZAZIONE CORRETTA")
print(f"   Statistiche usate (da X_train):")
print(f"      mean[0]: {mean_x[0]:.6f}")
print(f"      std[0]:  {std_x[0]:.6f}")
print(f"\n   X_train_norm (dovrebbe essere ~0 mean, ~1 std):")
print(f"      mean[0]: {X_train_norm[:, 0].mean():.6f}")
print(f"      std[0]:  {X_train_norm[:, 0].std():.6f}")
print(f"\n   X_val_norm (NON dovrebbe essere ~0 mean se fatto correttamente):")
print(f"      mean[0]: {X_val_norm[:, 0].mean():.6f}")
print(f"      std[0]:  {X_val_norm[:, 0].std():.6f}")

# Interpretazione
val_mean_diff = abs(X_val_norm[:, 0].mean())
if val_mean_diff < 0.01:
    print(f"\n⚠️  WARNING: X_val mean troppo vicino a 0 ({val_mean_diff:.6f})")
    print(f"   Possibile DATA LEAKAGE!")
else:
    print(f"\n✅ OK: X_val mean sufficientemente diverso da 0 ({val_mean_diff:.6f})")
    print(f"   Nessun data leakage rilevato.")

# -------------------------------------------------------------------
print("\n" + "="*70)
print("TEST 2: Verifica ERRATA (con data leakage - per confronto)")
print("="*70)

# Reload fresh data
X, y = load_cup('data/CUP/ML-CUP25-TR.csv', training=True)

# ERRORE: normalizza PRIMA dello split
X_wrong, mean_wrong, std_wrong = normalize(X)
X_train_wrong, X_val_wrong = train_test_split(X_wrong, test_size=0.2, random_state=42)

print(f"\n❌ NORMALIZZAZIONE ERRATA (prima dello split)")
print(f"   X_train_wrong mean[0]: {X_train_wrong[:, 0].mean():.6f}")
print(f"   X_val_wrong mean[0]:   {X_val_wrong[:, 0].mean():.6f}")
print(f"   ⚠️  Entrambi ~0 → DATA LEAKAGE PRESENTE!")

# -------------------------------------------------------------------
print("\n" + "="*70)
print("TEST 3: Verifica K-Fold (simulazione di 1 fold)")
print("="*70)

from sklearn.model_selection import KFold

X, y = load_cup('data/CUP/ML-CUP25-TR.csv', training=True)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

train_idx, val_idx = next(kf.split(X))
X_train_kf = X[train_idx]
X_val_kf = X[val_idx]

# Normalizzazione per-fold
X_train_kf_norm, mean_kf, std_kf = normalize(X_train_kf)
X_val_kf_norm, _, _ = normalize(X_val_kf, mean=mean_kf, std=std_kf)

print(f"\n✅ K-FOLD FOLD 1")
print(f"   X_train_kf_norm mean[0]: {X_train_kf_norm[:, 0].mean():.6f}")
print(f"   X_val_kf_norm mean[0]:   {X_val_kf_norm[:, 0].mean():.6f}")
print(f"   Stats from fold train mean[0]: {mean_kf[0]:.6f}")

# -------------------------------------------------------------------
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print("✅ Se X_val mean ≠ 0 → Normalizzazione corretta (no leakage)")
print("❌ Se X_val mean ≈ 0 → Data leakage (statistiche calcolate su tutto il dataset)")
print("="*70)


TEST 1: Verifica Data Leakage con Dataset CUP

📊 DATASET ORIGINALE (prima di split e normalizzazione)
   X shape: (500, 12)
   X mean (feature 0): 2.192591
   X std (feature 0):  10.276026
   X min/max: [-22.966, 29.436]

📦 DOPO SPLIT (ancora non normalizzato)
   X_train shape: (400, 12)
   X_val shape:   (100, 12)
   X_train mean (feature 0): 2.314169
   X_val mean (feature 0):   1.706282

✅ DOPO NORMALIZZAZIONE CORRETTA
   Statistiche usate (da X_train):
      mean[0]: 2.314169
      std[0]:  10.227198

   X_train_norm (dovrebbe essere ~0 mean, ~1 std):
      mean[0]: -0.000000
      std[0]:  1.000000

   X_val_norm (NON dovrebbe essere ~0 mean se fatto correttamente):
      mean[0]: -0.059438
      std[0]:  1.022268

✅ OK: X_val mean sufficientemente diverso da 0 (0.059438)
   Nessun data leakage rilevato.

TEST 2: Verifica ERRATA (con data leakage - per confronto)

❌ NORMALIZZAZIONE ERRATA (prima dello split)
   X_train_wrong mean[0]: 0.011831
   X_val_wrong mean[0]:   -0.047325
  

In [1]:
# TEST: Verifica se set_state() causa problemi con optimizer

import numpy as np
from nn.model import Model
from nn.layers import Dense
from nn.activations import ReLU, Sigmoid
from nn.losses import BinaryCrossEntropy
from nn.optim import Adam
from training.trainer import Trainer

# 1. Crea modello semplice
np.random.seed(42)
model = Model(
    modules=[
        Dense(10, 5, seed=42),
        ReLU(),
        Dense(5, 1, seed=42),
        Sigmoid()
    ],
    loss=BinaryCrossEntropy(),
    optimizer=Adam(lr=0.01)
)

# 2. Simula training
X_train = np.random.randn(100, 10)
y_train = np.random.randint(0, 2, (100, 1))

trainer = Trainer(model, verbose=0)

# Train 20 epoche
history = trainer.fit(X_train, y_train, epochs=20, batch_size=32)

# 3. Salva stato epoca 20
state_epoch_20 = model.get_state()

# Stampa stato optimizer PRIMA del restore
print("=" * 70)
print("PRIMA del restore (epoca 20):")
print(f"   Adam timestep (t): {model.optimizer.t}")
print(f"   Numero di parametri tracciati: {len(model.optimizer.m)}")
first_param_id = next(iter(model.optimizer.m.keys()))
print(f"   m[primo_param] mean: {model.optimizer.m[first_param_id].mean():.6f}")
print(f"   v[primo_param] mean: {model.optimizer.v[first_param_id].mean():.6f}")

# 4. Train altre 30 epoche
history2 = trainer.fit(X_train, y_train, epochs=30, batch_size=32)

print("\n" + "=" * 70)
print("DOPO 30 epoche aggiuntive (totale epoca 50):")
print(f"   Adam timestep (t): {model.optimizer.t}")
print(f"   m[primo_param] mean: {model.optimizer.m[first_param_id].mean():.6f}")
print(f"   v[primo_param] mean: {model.optimizer.v[first_param_id].mean():.6f}")

# 5. Restore stato epoca 20 (simula early stopping)
model.set_state(state_epoch_20)

print("\n" + "=" * 70)
print("DOPO set_state() - PROBLEMA CRITICO:")
print(f"   Pesi ripristinati: epoca 20 ✅")
print(f"   Adam timestep (t): {model.optimizer.t}")  # DOVREBBE essere 20, non 50!
print(f"   m[primo_param] mean: {model.optimizer.m[first_param_id].mean():.6f}")
print(f"   ⚠️  Optimizer state NON ripristinato!")

# 6. Verifica impatto su next step
y_pred = model.forward(X_train[:1], training=True)
loss = model.compute_loss(y_train[:1], y_pred)
dY = model.loss.backward(y_pred, y_train[:1])
model.backward(dY)

# Guarda la magnitude del gradient step
first_param = next(model.parameters())
grad_magnitude = np.abs(first_param).mean()

print(f"\n   Gradient magnitude prima di step: {grad_magnitude:.6f}")

model.step()

first_param_after = next(model.parameters())
update_magnitude = np.abs(first_param - first_param_after).mean()

print(f"   Update magnitude dopo step: {update_magnitude:.6f}")
print(f"   Ratio (update/param): {update_magnitude/grad_magnitude:.6f}")

print("\n" + "=" * 70)
print("INTERPRETAZIONE:")
print("   Se ratio >> 1 → Update troppo grande → Possibile divergenza")
print("=" * 70)


PRIMA del restore (epoca 20):
   Adam timestep (t): 80
   Numero di parametri tracciati: 4
   m[primo_param] mean: -0.001041
   v[primo_param] mean: 0.000165

DOPO 30 epoche aggiuntive (totale epoca 50):
   Adam timestep (t): 200
   m[primo_param] mean: -0.001656
   v[primo_param] mean: 0.000445

DOPO set_state() - PROBLEMA CRITICO:
   Pesi ripristinati: epoca 20 ✅
   Adam timestep (t): 80
   m[primo_param] mean: -0.001041
   ⚠️  Optimizer state NON ripristinato!

   Gradient magnitude prima di step: 0.280202
   Update magnitude dopo step: 0.000000
   Ratio (update/param): 0.000000

INTERPRETAZIONE:
   Se ratio >> 1 → Update troppo grande → Possibile divergenza


In [1]:
# ============================================================================
# TEST FINALE: Stessi Parametri e Approccio del Compagno
# ============================================================================

from data_handler.data_loader import load_cup, normalize, denormalize
from nn.model import Model
from nn.layers import Dense
from nn.activations import Tanh, Identity, ReLU
from nn.losses import MEE
from nn.optim import SGDMomentum
from nn.metrics import MEE as MEE_Metric
from nn.regularizers import L2
from nn.dropout import Dropout
from nn.callbacks import EarlyStopping
from training.trainer import Trainer
from training.kfold_cv import kfold_cross_validation
import numpy as np
from sklearn.model_selection import train_test_split

print("="*80)
print("FASE 1: PREPARAZIONE DATI (Split 90/10)")
print("="*80)

# --- CARICA DATI COMPLETI ---
X_full, y_full = load_cup('data/CUP/ML-CUP25-TR.csv', training=True)
print(f"\nDataset completo: X={X_full.shape}, y={y_full.shape}")

# --- SPLIT 90/10 (come il tuo compagno) ---
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_full, y_full, 
    test_size=0.10,  # 10% per test
    random_state=42, 
    shuffle=True
)

print(f"Train 90%: X={X_train_full.shape}, y={y_train_full.shape}")
print(f"Test 10%:  X={X_test.shape}, y={y_test.shape}")

# --- NORMALIZZAZIONE SUL TEST SET (usando stats del train) ---
# La normalizzazione sul train verrà fatta internamente dalla CV
X_train_full_raw = X_train_full.copy()  # Salviamo copia raw per CV
y_train_full_raw = y_train_full.copy()

X_train_full_norm, mean_X, std_X = normalize(X_train_full)
X_test_norm, _, _ = normalize(X_test, mean=mean_X, std=std_X)

y_train_full_norm, mean_y, std_y = normalize(y_train_full)
y_test_norm, _, _ = normalize(y_test, mean=mean_y, std=std_y)

print(f"\nStatistiche normalizzazione:")
print(f"X: mean={mean_X.mean():.4f}, std={std_X.mean():.4f}")
print(f"y: mean={mean_y.mean():.4f}, std={std_y.mean():.4f}")


# ============================================================================
print("\n" + "="*80)
print("FASE 2: 5-FOLD CROSS VALIDATION (per trovare avg_best_epoch)")
print("="*80)

# --- COSTRUISCI MODELLO E TRAINER PER CV ---
model_cv = Model(
    modules=[
        Dense(12, 64, seed=42),
        ReLU(),
        Dropout(p=0.2, seed=42),
        Dense(64, 32, seed=42),
        ReLU(),
        Dense(32, 16, seed=42),
        ReLU(),
        Dense(16, 4, seed=42),
        Identity(),
    ],
    loss=MEE(),
    optimizer=SGDMomentum(lr=0.05, momentum=0.8),  # Parametri del compagno
    regularizer=L2(lam=1e-4),                      # L2 modificato
    metrics=[MEE_Metric()],
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=250,
            min_delta=1e-4,
            mode="min",
            restore_best_weights=True,
            verbose=0  # Silenzioso durante CV
        )
    ]
)

trainer_cv = Trainer(model_cv, verbose=1)

print("\nParametri CV:")
print(f"  - K-Folds: 5")
print(f"  - Max Epochs: 1000")
print(f"  - Batch Size: 360")
print(f"  - LR: 0.01")
print(f"  - Momentum: 0.8")
print(f"  - L2 Lambda: 1e-4")
print(f"  - Early Stopping: patience=50, min_delta=1e-4")

# --- ESEGUI 5-FOLD CV ---
print("\n🔄 Avvio 5-Fold Cross Validation...\n")

train_stats, val_stats, histories, fold_results = kfold_cross_validation(
    X=X_train_full_raw,  # Dati NON normalizzati
    y=y_train_full_raw,
    model=model_cv,
    trainer=trainer_cv,
    k=5,
    epochs=1000,          # Max epoche
    batch_size=360,       # Batch del compagno
    shuffle=True,
    seed=42,
    verbose=1,
    include_reg_in_val=False,
    normalize_data=True,      # Normalizza X internamente
    normalize_target=True     # Normalizza y internamente
)

# --- ESTRAI BEST EPOCHS DA OGNI FOLD ---
best_epochs = []
for i, history in enumerate(histories):
    h_dict = history.to_dict()
    # Trova l'epoca con val_loss minima
    val_losses = h_dict['val_loss']
    best_epoch = np.argmin(val_losses) + 1  # +1 perché epoch parte da 1
    best_epochs.append(best_epoch)
    print(f"Fold {i+1}: Best epoch = {best_epoch}, Val Loss = {min(val_losses):.4f}")

avg_best_epoch = int(np.mean(best_epochs))

print("\n" + "="*80)
print("RISULTATI CROSS VALIDATION:")
print("="*80)
print(f"Best epochs per fold: {best_epochs}")
print(f"📊 AVG_BEST_EPOCH: {avg_best_epoch}")
print(f"\nTrain MEE: {train_stats['MEE']['mean']:.4f} ± {train_stats['MEE']['std']:.4f}")
print(f"Val MEE:   {val_stats['MEE']['mean']:.4f} ± {val_stats['MEE']['std']:.4f}")
print("="*80)


# ============================================================================
print("\n" + "="*80)
print(f"FASE 3: TRAINING FINALE (con {avg_best_epoch} epoche)")
print("="*80)

# --- RICOSTRUISCI MODELLO FINALE (SENZA EARLY STOPPING) ---
model_final = Model(
    modules=[
        Dense(12, 64, seed=42),
        ReLU(),
        Dropout(p=0.2, seed=42),
        Dense(64, 32, seed=42),
        ReLU(),
        Dense(32, 16, seed=42),
        ReLU(),
        Dense(16, 4, seed=42),
        Identity(),
    ],
    loss=MEE(),
    optimizer=SGDMomentum(lr=0.05, momentum=0.8),
    regularizer=L2(lam=1e-4),
    metrics=[MEE_Metric()],
    callbacks=[]  # NO early stopping per training finale
)

print("\n🚀 Training modello finale su tutto il train set (90%)...")

trainer_final = Trainer(model_final, verbose=1)
history_final = trainer_final.fit(
    X_train_full_norm, y_train_full_norm,
    X_val=None, y_val=None,  # NO validation set nel training finale
    epochs=avg_best_epoch,    # Usa avg_best_epoch dalla CV
    batch_size=360,
    shuffle=True,
    seed=42,
    include_reg_in_val=False
)

print(f"\n✅ Training completato ({avg_best_epoch} epoche)")


# ============================================================================
print("\n" + "="*80)
print("FASE 4: VALUTAZIONE SUL TEST SET (10%)")
print("="*80)

# --- FUNZIONE DI VALUTAZIONE DENORMALIZZATA ---
def evaluate_denormalized(X, y, model, mean_y, std_y):
    """Valuta il modello in scala originale"""
    y_pred = model.predict_proba(X)
    y_pred_denorm = denormalize(y_pred, mean_y, std_y)
    y_denorm = denormalize(y, mean_y, std_y)
    
    mee = np.mean(np.linalg.norm(y_pred_denorm - y_denorm, axis=1))
    mse = np.mean(np.sum((y_pred_denorm - y_denorm)**2, axis=1))
    
    return {'MEE': mee, 'MSE': mse}

# --- VALUTAZIONE TRAIN ---
print("\n📊 Metriche Train Set (90%):")
train_metrics = evaluate_denormalized(
    X_train_full_norm, y_train_full_norm, 
    model_final, mean_y, std_y
)
print(f"  MEE: {train_metrics['MEE']:.4f}")
print(f"  MSE: {train_metrics['MSE']:.4f}")

# --- VALUTAZIONE TEST ---
print("\n📊 Metriche Test Set (10%):")
test_metrics = evaluate_denormalized(
    X_test_norm, y_test_norm, 
    model_final, mean_y, std_y
)
print(f"  MEE: {test_metrics['MEE']:.4f} ⭐")
print(f"  MSE: {test_metrics['MSE']:.4f}")

print("\n" + "="*80)
print("TEST COMPLETATO")
print("="*80)
print(f"\n🎯 RISULTATO FINALE: MEE = {test_metrics['MEE']:.2f} (scala originale)")
print(f"   (Compagno ha ottenuto: MEE ≈ 23.36)")
print("\n💡 Prossimo step: Usa lo script di ensemble per migliorare ulteriormente!")


# ============================================================================
print("\n" + "="*80)
print("SANITY CHECK - Predizioni su 3 samples del test")
print("="*80)

X_sample = X_test_norm[:3]
y_sample = y_test_norm[:3]
y_pred = model_final.predict_proba(X_sample)

y_pred_denorm = denormalize(y_pred, mean_y, std_y)
y_sample_denorm = denormalize(y_sample, mean_y, std_y)

for i in range(3):
    error = np.linalg.norm(y_pred_denorm[i] - y_sample_denorm[i])
    print(f"\nSample {i+1}:")
    print(f"  True:      {y_sample_denorm[i]}")
    print(f"  Predicted: {y_pred_denorm[i]}")
    print(f"  Error:     {error:.4f}")

print("\n" + "="*80)
print("🎉 MODELLO PRONTO! Ora puoi lanciare l'ensemble")
print("="*80)

FASE 1: PREPARAZIONE DATI (Split 90/10)

Dataset completo: X=(500, 12), y=(500, 4)
Train 90%: X=(450, 12), y=(450, 4)
Test 10%:  X=(50, 12), y=(50, 4)

Statistiche normalizzazione:
X: mean=1.0896, std=9.5079
y: mean=0.1507, std=18.5519

FASE 2: 5-FOLD CROSS VALIDATION (per trovare avg_best_epoch)

Parametri CV:
  - K-Folds: 5
  - Max Epochs: 1000
  - Batch Size: 360
  - LR: 0.01
  - Momentum: 0.8
  - L2 Lambda: 1e-4
  - Early Stopping: patience=50, min_delta=1e-4

🔄 Avvio 5-Fold Cross Validation...

Task: Regression (y.shape=(450, 4))
Using KFold with k=5

Fold 1/5
------------------------------------------------------------

      _____     ___                 _ _ _ 
      \_   \   / __\__ ___   ____ _| | (_)
       / /\/  / /  / _` \ \ / / _` | | | |
    /\/ /_   / /__| (_| |\ V / (_| | | | |
    \____/   \____/\__,_| \_/ \__,_|_|_|_|
        
Epoch 1000/1000 [==============================>] loss: 0.9630 MEE: 0.9550 val_loss: 1.1121 val_MEE: 1.1121
  Train: loss=0.8471, MEE=0.8471
 